# Clinician Rating Analysis — Fareez RAG Summarization

Statistical analysis of clinician ratings for 160 AI-generated medical summaries (4 models x 40 Fareez OSCE conversations).

**Raters:** 3 physician fellows (ER, IR, GI) rated each summary on 6 dimensions using 5-point Likert scales adapted from PDQI-9 and QNOTE.

**Models:** GPT-OSS 120B, GPT-OSS 20B, Qwen3 32B (Groq API), MedGemma 4B-IT (Modal A10G GPU)

In [1]:
# Papermill parameters cell
base_dir = '.'

In [2]:
# Parameters
base_dir = "."


In [3]:
import os
import numpy as np
import pandas as pd
from scipy import stats
import krippendorff
from itertools import combinations
from IPython.display import display, Markdown

DIMENSIONS = ['accuracy', 'completeness', 'organization', 'conciseness',
              'clinical_utility', 'overall_quality']
DIM_LABELS = {'accuracy': 'Accuracy', 'completeness': 'Completeness',
              'organization': 'Organization', 'conciseness': 'Conciseness',
              'clinical_utility': 'Clinical Utility', 'overall_quality': 'Overall Quality'}
RATER_MAP = {'rater_1': 'ER Fellow', 'rater_2': 'IR Fellow', 'rater_3': 'GI Fellow'}
PACKETS_DIR = os.path.join(base_dir, 'rating_packets')

## 1. Load and Merge Data

In [4]:
key = pd.read_csv(os.path.join(PACKETS_DIR, 'ANSWER_KEY_DO_NOT_SHARE.csv'))

frames = []
for rater_id in ['rater_1', 'rater_2', 'rater_3']:
    df = pd.read_csv(os.path.join(PACKETS_DIR, f'ratings_{rater_id}.csv'))
    df['rater'] = rater_id
    frames.append(df)

merged = pd.concat(frames, ignore_index=True)
merged = merged.merge(key[['summary_id', 'conversation', 'model', 'specialty', 'condition']],
                      on='summary_id', how='left')
for d in DIMENSIONS:
    merged[d] = pd.to_numeric(merged[d], errors='coerce')
merged['item'] = merged['conversation'] + '_' + merged['model']

n_ratings = len(merged)
n_raters = merged['rater'].nunique()
n_models = merged['model'].nunique()
n_convos = merged['conversation'].nunique()
spec_counts = merged.groupby('specialty')['conversation'].nunique().to_dict()

print(f'Total ratings: {n_ratings} ({n_raters} raters x {n_models} models x {n_convos} conversations)')
print(f'Models: {sorted(merged["model"].unique())}')
print(f'Specialties: {spec_counts}')

Total ratings: 480 (3 raters x 4 models x 40 conversations)
Models: ['gpt-oss-120b', 'gpt-oss-20b', 'medgemma-4b', 'qwen3-32b']
Specialties: {'CAR': 4, 'DER': 1, 'GAS': 5, 'MSK': 10, 'RES': 20}


## 2. Rater Profiles and Mean Scores

In [5]:
rows = []
for r in ['rater_1', 'rater_2', 'rater_3']:
    rd = merged[merged['rater'] == r]
    row = {'Rater': RATER_MAP[r]}
    for d in DIMENSIONS:
        row[DIM_LABELS[d]] = f"{rd[d].mean():.2f}"
    rows.append(row)

# Grand mean
row = {'Rater': '**Grand Mean**'}
for d in DIMENSIONS:
    row[DIM_LABELS[d]] = f"**{merged[d].mean():.2f}**"
rows.append(row)

rater_df = pd.DataFrame(rows).set_index('Rater')
display(Markdown('### Table 1: Mean Scores by Rater (across all 160 summaries)'))
display(Markdown(rater_df.to_markdown()))

### Table 1: Mean Scores by Rater (across all 160 summaries)

| Rater          | Accuracy   | Completeness   | Organization   | Conciseness   | Clinical Utility   | Overall Quality   |
|:---------------|:-----------|:---------------|:---------------|:--------------|:-------------------|:------------------|
| ER Fellow      | 2.91       | 3.71           | 4.59           | 2.84          | 3.41               | 3.21              |
| IR Fellow      | 2.62       | 3.98           | 4.84           | 4.39          | 3.36               | 3.30              |
| GI Fellow      | 2.11       | 3.46           | 4.71           | 3.73          | 3.01               | 2.94              |
| **Grand Mean** | **2.55**   | **3.71**       | **4.71**       | **3.65**      | **3.26**           | **3.15**          |

## 3. Pairwise Inter-Rater Reliability (Krippendorff's Alpha)

In [6]:
raters = ['rater_1', 'rater_2', 'rater_3']
pairs = [('rater_1', 'rater_2'), ('rater_1', 'rater_3'), ('rater_2', 'rater_3')]
pair_labels = ['ER vs IR', 'ER vs GI', 'IR vs GI']

alpha_rows = []
for (r1, r2), plabel in zip(pairs, pair_labels):
    row = {'Rater Pair': plabel}
    alphas = []
    for d in DIMENSIONS:
        sub = merged[merged['rater'].isin([r1, r2])]
        pivot = sub.pivot_table(index='item', columns='rater', values=d, aggfunc='first')
        data = pivot[[r1, r2]].values.T
        a = krippendorff.alpha(data, level_of_measurement='ordinal')
        alphas.append(a)
        marker = ' \u2713' if a >= 0.667 else ' \u25cb' if a >= 0.4 else ''
        row[DIM_LABELS[d]] = f"{a:.3f}{marker}"
    row['Average'] = f"{np.mean(alphas):.3f}"
    alpha_rows.append(row)

alpha_df = pd.DataFrame(alpha_rows).set_index('Rater Pair')
display(Markdown('### Table 2: Pairwise Krippendorff\'s Alpha (ordinal)'))
display(Markdown('> \u2713 = acceptable (\u2265 0.667) &nbsp; \u25cb = fair (\u2265 0.400)'))
display(Markdown(alpha_df.to_markdown()))

### Table 2: Pairwise Krippendorff's Alpha (ordinal)

> ✓ = acceptable (≥ 0.667) &nbsp; ○ = fair (≥ 0.400)

| Rater Pair   |   Accuracy | Completeness   | Organization   |   Conciseness | Clinical Utility   | Overall Quality   |   Average |
|:-------------|-----------:|:---------------|:---------------|--------------:|:-------------------|:------------------|----------:|
| ER vs IR     |      0.164 | 0.408 ○        | 0.347          |        -0.455 | 0.431 ○            | 0.406 ○           |     0.217 |
| ER vs GI     |     -0.024 | 0.612 ○        | 0.790 ✓        |        -0.198 | 0.417 ○            | 0.513 ○           |     0.352 |
| IR vs GI     |      0.125 | 0.272          | 0.426 ○        |        -0.101 | 0.269              | 0.288             |     0.213 |

## 4. Pairwise Percent Agreement (within 1 point)

In [7]:
agree_rows = []
for (r1, r2), plabel in zip(pairs, pair_labels):
    row = {'Rater Pair': plabel}
    pcts = []
    for d in DIMENSIONS:
        sub = merged[merged['rater'].isin([r1, r2])]
        pivot = sub.pivot_table(index='item', columns='rater', values=d, aggfunc='first').dropna()
        v1, v2 = pivot[r1].values, pivot[r2].values
        pct = np.mean(np.abs(v1 - v2) <= 1) * 100
        pcts.append(pct)
        row[DIM_LABELS[d]] = f"{pct:.1f}%"
    row['Average'] = f"{np.mean(pcts):.1f}%"
    agree_rows.append(row)

agree_df = pd.DataFrame(agree_rows).set_index('Rater Pair')
display(Markdown('### Table 3: Percent Agreement Within 1 Point (5-point scale)'))
display(Markdown(agree_df.to_markdown()))

### Table 3: Percent Agreement Within 1 Point (5-point scale)

| Rater Pair   | Accuracy   | Completeness   | Organization   | Conciseness   | Clinical Utility   | Overall Quality   | Average   |
|:-------------|:-----------|:---------------|:---------------|:--------------|:-------------------|:------------------|:----------|
| ER vs IR     | 81.2%      | 94.4%          | 95.6%          | 43.8%         | 87.5%              | 83.1%             | 80.9%     |
| ER vs GI     | 91.2%      | 98.8%          | 98.8%          | 93.8%         | 97.5%              | 98.8%             | 96.5%     |
| IR vs GI     | 73.8%      | 93.8%          | 100.0%         | 92.5%         | 76.9%              | 76.2%             | 85.5%     |

## 5. Model Comparison: Descriptive Statistics

In [8]:
model_rows = []
for model in ['gpt-oss-120b', 'gpt-oss-20b', 'qwen3-32b', 'medgemma-4b']:
    md = merged[merged['model'] == model]
    row = {'Model': model}
    dim_means = []
    for d in DIMENSIONS:
        m = md[d].mean()
        s = md[d].std()
        dim_means.append(m)
        row[DIM_LABELS[d]] = f"{m:.2f} ({s:.2f})"
    row['Composite'] = f"{np.mean(dim_means):.2f}"
    model_rows.append(row)

model_df = pd.DataFrame(model_rows).set_index('Model')
display(Markdown('### Table 4: Mean (SD) per Model per Dimension (n=120 ratings per cell: 40 conversations x 3 raters)'))
display(Markdown(model_df.to_markdown()))

### Table 4: Mean (SD) per Model per Dimension (n=120 ratings per cell: 40 conversations x 3 raters)

| Model        | Accuracy    | Completeness   | Organization   | Conciseness   | Clinical Utility   | Overall Quality   |   Composite |
|:-------------|:------------|:---------------|:---------------|:--------------|:-------------------|:------------------|------------:|
| gpt-oss-120b | 2.70 (0.90) | 4.14 (0.57)    | 5.00 (0.00)    | 3.64 (0.99)   | 3.54 (0.88)        | 3.44 (0.86)       |        3.74 |
| gpt-oss-20b  | 2.62 (0.92) | 3.92 (0.64)    | 5.00 (0.00)    | 3.92 (0.78)   | 3.47 (0.85)        | 3.35 (0.86)       |        3.71 |
| qwen3-32b    | 2.50 (0.90) | 3.83 (0.62)    | 4.94 (0.27)    | 3.70 (0.71)   | 3.42 (0.91)        | 3.29 (0.90)       |        3.61 |
| medgemma-4b  | 2.38 (0.81) | 2.97 (0.74)    | 3.91 (1.11)    | 3.35 (1.23)   | 2.62 (0.84)        | 2.51 (0.92)       |        2.95 |

## 6. Model Comparison: Friedman Test

In [9]:
models = ['gpt-oss-120b', 'gpt-oss-20b', 'qwen3-32b', 'medgemma-4b']

friedman_rows = []
for d in DIMENSIONS:
    avg_ratings = merged.groupby(['conversation', 'model'])[d].mean().reset_index()
    pivot = avg_ratings.pivot(index='conversation', columns='model', values=d).dropna()
    groups = [pivot[m].values for m in models if m in pivot.columns]
    stat, p = stats.friedmanchisquare(*groups)
    n = len(pivot)
    k = len(groups)
    w = stat / (n * (k - 1))
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    friedman_rows.append({
        'Dimension': DIM_LABELS[d],
        'Chi-squared': f"{stat:.3f}",
        'p-value': f"{p:.6f}",
        "Kendall's W": f"{w:.4f}",
        'Sig.': sig,
        'n': n
    })

friedman_df = pd.DataFrame(friedman_rows).set_index('Dimension')
display(Markdown('### Table 5: Friedman Test — Model Differences per Dimension'))
display(Markdown('> Rater scores averaged per (conversation, model) pair before testing. Kendall\'s W = effect size.'))
display(Markdown(friedman_df.to_markdown()))

### Table 5: Friedman Test — Model Differences per Dimension

> Rater scores averaged per (conversation, model) pair before testing. Kendall's W = effect size.

| Dimension        |   Chi-squared |   p-value |   Kendall's W | Sig.   |   n |
|:-----------------|--------------:|----------:|--------------:|:-------|----:|
| Accuracy         |         5.758 |  0.123993 |        0.048  | ns     |  40 |
| Completeness     |        89.303 |  0        |        0.7442 | ***    |  40 |
| Organization     |       113.51  |  0        |        0.9459 | ***    |  40 |
| Conciseness      |        15.056 |  0.001769 |        0.1255 | **     |  40 |
| Clinical Utility |        51.698 |  0        |        0.4308 | ***    |  40 |
| Overall Quality  |        48.111 |  0        |        0.4009 | ***    |  40 |

## 7. Pairwise Model Comparison: Wilcoxon Signed-Rank (Bonferroni)

In [10]:
model_pairs = list(combinations(models, 2))
n_comp = len(model_pairs)
alpha_corrected = 0.05 / n_comp

display(Markdown(f'> {n_comp} pairwise comparisons, Bonferroni-corrected alpha = {alpha_corrected:.4f}'))

for d in DIMENSIONS:
    avg_ratings = merged.groupby(['conversation', 'model'])[d].mean().reset_index()
    pivot = avg_ratings.pivot(index='conversation', columns='model', values=d).dropna()

    pw_rows = []
    for m1, m2 in model_pairs:
        if m1 not in pivot.columns or m2 not in pivot.columns:
            continue
        v1, v2 = pivot[m1].values, pivot[m2].values
        try:
            stat, p = stats.wilcoxon(v1, v2, alternative='two-sided')
            diff = np.mean(v1) - np.mean(v2)
            sig = '***' if p < 0.001 else '**' if p < alpha_corrected else '*' if p < 0.05 else 'ns'
            pw_rows.append({
                'Pair': f"{m1} vs {m2}",
                'W': f"{stat:.1f}",
                'p': f"{p:.6f}",
                'Mean Diff': f"{diff:+.3f}",
                'Sig.': sig
            })
        except Exception:
            pw_rows.append({'Pair': f"{m1} vs {m2}", 'W': 'N/A', 'p': 'N/A', 'Mean Diff': 'N/A', 'Sig.': ''})

    pw_df = pd.DataFrame(pw_rows).set_index('Pair')
    display(Markdown(f'#### {DIM_LABELS[d]}'))
    display(Markdown(pw_df.to_markdown()))

> 6 pairwise comparisons, Bonferroni-corrected alpha = 0.0083

#### Accuracy

| Pair                        |     W |        p |   Mean Diff | Sig.   |
|:----------------------------|------:|---------:|------------:|:-------|
| gpt-oss-120b vs gpt-oss-20b | 166   | 0.165304 |       0.075 | ns     |
| gpt-oss-120b vs qwen3-32b   |  93.5 | 0.020817 |       0.2   | *      |
| gpt-oss-120b vs medgemma-4b |  53   | 0.003108 |       0.325 | **     |
| gpt-oss-20b vs qwen3-32b    | 163.5 | 0.240541 |       0.125 | ns     |
| gpt-oss-20b vs medgemma-4b  | 153.5 | 0.038037 |       0.25  | *      |
| qwen3-32b vs medgemma-4b    | 136.5 | 0.204663 |       0.125 | ns     |

#### Completeness

| Pair                        |     W |        p |   Mean Diff | Sig.   |
|:----------------------------|------:|---------:|------------:|:-------|
| gpt-oss-120b vs gpt-oss-20b |  41.5 | 0.00058  |       0.217 | ***    |
| gpt-oss-120b vs qwen3-32b   |  26   | 3e-05    |       0.317 | ***    |
| gpt-oss-120b vs medgemma-4b |   0   | 0        |       1.175 | ***    |
| gpt-oss-20b vs qwen3-32b    | 155.5 | 0.176547 |       0.1   | ns     |
| gpt-oss-20b vs medgemma-4b  |   0   | 0        |       0.958 | ***    |
| qwen3-32b vs medgemma-4b    |   0   | 0        |       0.858 | ***    |

...

| Pair                        |     W |        p |   Mean Diff | Sig.   |
|:----------------------------|------:|---------:|------------:|:-------|
| gpt-oss-120b vs gpt-oss-20b |  40   | 0.000491 |      -0.275 | ***    |
| gpt-oss-120b vs qwen3-32b   | 196.5 | 0.881097 |      -0.058 | ns     |
| gpt-oss-120b vs medgemma-4b | 199.5 | 0.339949 |       0.292 | ns     |
| gpt-oss-20b vs qwen3-32b    |  70   | 0.001848 |       0.217 | **     |
| gpt-oss-20b vs medgemma-4b  | 126   | 0.003007 |       0.567 | **     |
| qwen3-32b vs medgemma-4b    | 160.5 | 0.331122 |       0.35  | ns     |

#### Clinical Utility

| Pair                        |     W |        p |   Mean Diff | Sig.   |
|:----------------------------|------:|---------:|------------:|:-------|
| gpt-oss-120b vs gpt-oss-20b | 130.5 | 0.250192 |       0.075 | ns     |
| gpt-oss-120b vs qwen3-32b   | 140   | 0.091454 |       0.125 | ns     |
| gpt-oss-120b vs medgemma-4b |  13   | 1e-06    |       0.925 | ***    |
| gpt-oss-20b vs qwen3-32b    | 220   | 0.58149  |       0.05  | ns     |
| gpt-oss-20b vs medgemma-4b  |  16.5 | 1e-06    |       0.85  | ***    |
| qwen3-32b vs medgemma-4b    |  31   | 2e-06    |       0.8   | ***    |

#### Overall Quality

| Pair                        |     W |        p |   Mean Diff | Sig.   |
|:----------------------------|------:|---------:|------------:|:-------|
| gpt-oss-120b vs gpt-oss-20b | 180   | 0.274377 |       0.092 | ns     |
| gpt-oss-120b vs qwen3-32b   | 114   | 0.041612 |       0.15  | *      |
| gpt-oss-120b vs medgemma-4b |  16   | 1e-06    |       0.933 | ***    |
| gpt-oss-20b vs qwen3-32b    | 221.5 | 0.423668 |       0.058 | ns     |
| gpt-oss-20b vs medgemma-4b  |  25.5 | 1e-06    |       0.842 | ***    |
| qwen3-32b vs medgemma-4b    |  40.5 | 4e-06    |       0.783 | ***    |

## 8. Specialty Subanalysis: Friedman Test (RES, MSK)

In [11]:
for specialty in ['RES', 'MSK']:
    sd = merged[merged['specialty'] == specialty]
    n_c = sd['conversation'].nunique()
    display(Markdown(f'#### {specialty} ({n_c} conversations)'))

    spec_rows = []
    for d in DIMENSIONS:
        avg_ratings = sd.groupby(['conversation', 'model'])[d].mean().reset_index()
        pivot = avg_ratings.pivot(index='conversation', columns='model', values=d).dropna()
        if len(pivot) < 5:
            continue
        groups = [pivot[m].values for m in models if m in pivot.columns]
        stat, p = stats.friedmanchisquare(*groups)
        n = len(pivot)
        k = len(groups)
        w = stat / (n * (k - 1))
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        spec_rows.append({
            'Dimension': DIM_LABELS[d],
            'Chi-squared': f"{stat:.3f}",
            'p-value': f"{p:.6f}",
            "Kendall's W": f"{w:.4f}",
            'Sig.': sig
        })

    spec_df = pd.DataFrame(spec_rows).set_index('Dimension')
    display(Markdown(spec_df.to_markdown()))

#### RES (20 conversations)

| Dimension        |   Chi-squared |   p-value |   Kendall's W | Sig.   |
|:-----------------|--------------:|----------:|--------------:|:-------|
| Accuracy         |         9.863 |  0.01977  |        0.1644 | *      |
| Completeness     |        44.19  |  0        |        0.7365 | ***    |
| Organization     |        57.429 |  0        |        0.9571 | ***    |
| Conciseness      |        10.604 |  0.014073 |        0.1767 | *      |
| Clinical Utility |        33.034 |  0        |        0.5506 | ***    |
| Overall Quality  |        30.249 |  1e-06    |        0.5041 | ***    |

#### MSK (10 conversations)

| Dimension        |   Chi-squared |   p-value |   Kendall's W | Sig.   |
|:-----------------|--------------:|----------:|--------------:|:-------|
| Accuracy         |         1.292 |  0.731113 |        0.0431 | ns     |
| Completeness     |        24.233 |  2.2e-05  |        0.8078 | ***    |
| Organization     |        30     |  1e-06    |        1      | ***    |
| Conciseness      |         3.296 |  0.348158 |        0.1099 | ns     |
| Clinical Utility |        10.354 |  0.015787 |        0.3451 | *      |
| Overall Quality  |         9.072 |  0.028345 |        0.3024 | *      |

## 9. Disagreement Analysis

In [12]:
dis_rows = []
for d in DIMENSIONS:
    pivot = merged.pivot_table(index='item', columns='rater', values=d, aggfunc='first').dropna()
    spreads = pivot.max(axis=1) - pivot.min(axis=1)
    n_items = len(pivot)
    n_spread2 = (spreads >= 2).sum()
    n_spread3 = (spreads >= 3).sum()
    dis_rows.append({
        'Dimension': DIM_LABELS[d],
        'Items (total)': n_items,
        'Spread >= 2 pts': f"{n_spread2} ({n_spread2/n_items*100:.1f}%)",
        'Spread >= 3 pts': f"{n_spread3} ({n_spread3/n_items*100:.1f}%)",
        'Mean Spread': f"{spreads.mean():.2f}",
    })

dis_df = pd.DataFrame(dis_rows).set_index('Dimension')
display(Markdown('### Table 6: Rater Disagreement by Dimension (max spread across 3 raters)'))
display(Markdown(dis_df.to_markdown()))

### Table 6: Rater Disagreement by Dimension (max spread across 3 raters)

| Dimension        |   Items (total) | Spread >= 2 pts   | Spread >= 3 pts   |   Mean Spread |
|:-----------------|----------------:|:------------------|:------------------|--------------:|
| Accuracy         |             160 | 63 (39.4%)        | 10 (6.2%)         |          1.37 |
| Completeness     |             160 | 16 (10.0%)        | 0 (0.0%)          |          0.8  |
| Organization     |             160 | 7 (4.4%)          | 0 (0.0%)          |          0.29 |
| Conciseness      |             160 | 91 (56.9%)        | 8 (5.0%)          |          1.57 |
| Clinical Utility |             160 | 46 (28.7%)        | 4 (2.5%)          |          1.14 |
| Overall Quality  |             160 | 46 (28.7%)        | 5 (3.1%)          |          1.11 |

## 10. Summary

**Key findings:**

1. **ER and GI fellows show the strongest pairwise agreement** (alpha 0.79 on organization, 0.61 on completeness, 96.5% agreement within 1 point)
2. **Conciseness has the poorest reliability** — the IR fellow rated systematically higher (mean 4.39 vs 2.84/3.73), likely reflecting different interpretations of the construct
3. **MedGemma 4B is significantly worse** than the three Groq models on 5 of 6 dimensions (Friedman p < 0.001)
4. **GPT-OSS 120B, GPT-OSS 20B, and Qwen3 32B are statistically indistinguishable** on most dimensions
5. **Organization is the strongest dimension** (mean 4.71/5.0) — RAG schema retrieval successfully enforces SOAP structure
6. **Accuracy is the weakest dimension** (mean 2.55/5.0) — all models struggle with EHR data integration, fabricating exam findings or importing mismatched patient data

In [13]:
# Save analysis outputs
output_dir = os.path.join(base_dir, 'results', 'fareez')
os.makedirs(output_dir, exist_ok=True)

merged.to_csv(os.path.join(output_dir, 'fareez_clinician_ratings.csv'), index=False)
print(f'Saved: {os.path.join(output_dir, "fareez_clinician_ratings.csv")}')

Saved: ./results/fareez/fareez_clinician_ratings.csv
